In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


# 🔁 Cell 1 – Setup & Imports

In [3]:
# Cell 1 — Spark + ML imports

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

print("Spark version:", spark.version)


Spark version: 3.5.1


# 📂 Cell 2 – Load policy churn feature splits from golddata

In [12]:
# Cell 2 — Load ft_policy_churn_split from golddata

STORAGE_ACCOUNT = "clientdatastorage"
CONTAINER_GOLD  = "golddata"

FT_POLICY_CHURN_SPLIT_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"ft_policy_churn_split"
)

print("Reading from:", FT_POLICY_CHURN_SPLIT_PATH)

churn_all = (
    spark.read
         .format("delta")
         .load(FT_POLICY_CHURN_SPLIT_PATH)
)

print("Total rows:", churn_all.count())
churn_all.groupBy("dataset_split").count().show()
churn_all.printSchema()
churn_all.show(5, truncate=False)


Reading from: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn_split
Total rows: 381109
+-------------+------+
|dataset_split| count|
+-------------+------+
|        train|304742|
|         test| 76367|
+-------------+------+

root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = true)
 |-- Channel: string (nullable = true)
 |-- Sum_Insured_GBP: double (nullable = true)
 |-- Annual_Premium_GBP: double (nullable = true)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: integer (nullable = true)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = true)
 |-- Premium_Band: string (nullable = true)
 |-- Discount_Band: string (nullable = true)
 |-- Renewal_Outcome: string (nullable = true)

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+------------------+-----------+-------------+-------------------------+-------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|Is_Renewal_Offered|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|dataset_split|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----

You should see train and test counts that match what you printed earlier.

# ✂️ Cell 3 – Train / Test split DataFrames

In [13]:
# Cell 3 — Separate train and test splits

train_df = churn_all.filter(F.col("dataset_split") == "train").cache()
test_df  = churn_all.filter(F.col("dataset_split") == "test").cache()

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

# Quick sanity check on label distribution
for name, df in [("train", train_df), ("test", test_df)]:
    print(f"\n{name} label distribution:")
    (
        df.groupBy("Churn_Label")
          .count()
          .orderBy("Churn_Label")
          .show()
    )


25/12/07 18:59:22 WARN CacheManager: Asked to cache already cached data.
25/12/07 18:59:22 WARN CacheManager: Asked to cache already cached data.


Train rows: 304742
Test rows : 76367

train label distribution:
+-----------+------+
|Churn_Label| count|
+-----------+------+
|       NULL| 76429|
|          0|159495|
|          1| 68818|
+-----------+------+


test label distribution:
+-----------+-----+
|Churn_Label|count|
+-----------+-----+
|       NULL|19241|
|          0|39820|
|          1|17306|
+-----------+-----+



# 🧱 Cell 4 – Feature engineering pipeline (index, encode, assemble)

We’ll treat Churn_Label as the target, and use a mix of numeric & categorical features.

In [14]:
# Cell 4 — Define features and build preprocessing pipeline

# 1) Choose feature columns (can tune later)
numeric_cols = [
    "Sum_Insured_GBP",
    "Annual_Premium_GBP",
    "Policy_Duration_Days",
    "Premium_per_1k_SumInsured"
]

categorical_cols = [
    "Product_Line",
    "Channel",
    "Tenure_Band",
    "Premium_Band",
    "Discount_Band",
    "Renewal_Outcome",
    "Is_Discounted"
]

label_col = "Churn_Label"

# 2) Basic null handling
def prep_nulls(df):
    df_num_filled = df
    for c in numeric_cols:
        if c in df_num_filled.columns:
            df_num_filled = df_num_filled.withColumn(
                c,
                F.coalesce(F.col(c), F.lit(0.0))
            )
    df_cat_filled = df_num_filled
    for c in categorical_cols:
        if c in df_cat_filled.columns:
            df_cat_filled = df_cat_filled.withColumn(
                c,
                F.coalesce(F.col(c).cast("string"), F.lit("Unknown"))
            )
    # ensure label is numeric 0/1
    df_final = df_cat_filled.withColumn(label_col, F.col(label_col).cast("double"))
    return df_final

# Apply null handling AND drop rows with missing/invalid labels
train_pre = (
    prep_nulls(train_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

test_pre = (
    prep_nulls(test_df)
    .filter(F.col(label_col).isNotNull())
    .filter(F.col(label_col).isin(0.0, 1.0))
)

print("After cleaning:")
print("  train_pre rows:", train_pre.count())
print("  test_pre rows :", test_pre.count())


# 3) Build indexers + one-hot encoders
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh"
    )
    for c in categorical_cols
]

# 4) VectorAssembler for all features
feature_cols = numeric_cols + [f"{c}_oh" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

print("Numeric features :", numeric_cols)
print("Categorical feats:", categorical_cols)
print("Total features col list length:", len(feature_cols))


After cleaning:
  train_pre rows: 228313
  test_pre rows : 57126
Numeric features : ['Sum_Insured_GBP', 'Annual_Premium_GBP', 'Policy_Duration_Days', 'Premium_per_1k_SumInsured']
Categorical feats: ['Product_Line', 'Channel', 'Tenure_Band', 'Premium_Band', 'Discount_Band', 'Renewal_Outcome', 'Is_Discounted']
Total features col list length: 11


# 🤖 Cell 5 – Define models (LR, RF, GBT) as Pipelines

In [15]:
# Cell 5 — Define three ML pipelines

# Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

# Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=100,
    maxDepth=8,
    subsamplingRate=0.8,
    featureSubsetStrategy="auto",
    seed=42
)

pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])

# Gradient Boosted Trees
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=label_col,
    maxIter=80,
    maxDepth=5,
    stepSize=0.05,
    seed=42
)

pipeline_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt])

print("Pipelines defined: pipeline_lr, pipeline_rf, pipeline_gbt")


Pipelines defined: pipeline_lr, pipeline_rf, pipeline_gbt


# 📊 Cell 6 – Helper to train & evaluate a model

In [17]:
# Cell 6 — Training + evaluation helper

def evaluate_binary(pred_df, label_col=label_col, prediction_col="prediction", prob_col="probability"):
    # AUC ROC
    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc_roc = evaluator_roc.evaluate(pred_df)

    # AUC PR
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=label_col,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )
    auc_pr = evaluator_pr.evaluate(pred_df)

    # F1 & Accuracy (treat as multiclass)
    f1_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="f1"
    )
    f1 = f1_eval.evaluate(pred_df)

    acc_eval = MulticlassClassificationEvaluator(
        labelCol=label_col,
        predictionCol=prediction_col,
        metricName="accuracy"
    )
    acc = acc_eval.evaluate(pred_df)

    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1": f1,
        "accuracy": acc
    }


def train_and_eval(pipeline, train_df, test_df, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    metrics = evaluate_binary(pred)

    print(f"{model_name} metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    return model, metrics, pred


# 🧪 Cell 7 – Train all three models & compare metrics

In [18]:
# Cell 7 — Train LR, RF, GBT and compare

results = []

lr_model, lr_metrics, lr_pred = train_and_eval(
    pipeline_lr, train_pre, test_pre, "LogisticRegression"
)
results.append(("LogisticRegression", lr_metrics))

rf_model, rf_metrics, rf_pred = train_and_eval(
    pipeline_rf, train_pre, test_pre, "RandomForest"
)
results.append(("RandomForest", rf_metrics))

gbt_model, gbt_metrics, gbt_pred = train_and_eval(
    pipeline_gbt, train_pre, test_pre, "GBTClassifier"
)
results.append(("GBTClassifier", gbt_metrics))

# Show as a small Spark table for convenience
metrics_rows = [
    (name,
     m["auc_roc"],
     m["auc_pr"],
     m["f1"],
     m["accuracy"])
    for name, m in results
]

metrics_df = spark.createDataFrame(
    metrics_rows,
    schema="model STRING, auc_roc DOUBLE, auc_pr DOUBLE, f1 DOUBLE, accuracy DOUBLE"
)

print("\nModel comparison:")
metrics_df.show(truncate=False)



===== Training LogisticRegression =====


25/12/07 18:59:47 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/07 18:59:47 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


Scoring test set with LogisticRegression ...
LogisticRegression metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000

===== Training RandomForest =====


Scoring test set with RandomForest ...
RandomForest metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000

===== Training GBTClassifier =====


Scoring test set with GBTClassifier ...
GBTClassifier metrics:
  auc_roc: 1.0000
  auc_pr: 1.0000
  f1: 1.0000
  accuracy: 1.0000

Model comparison:
+------------------+------------------+------------------+---+--------+
|model             |auc_roc           |auc_pr            |f1 |accuracy|
+------------------+------------------+------------------+---+--------+
|LogisticRegression|0.9999999201886358|0.9999988306726361|1.0|1.0     |
|RandomForest      |1.0               |1.0               |1.0|1.0     |
|GBTClassifier     |0.9999999999999999|1.0               |1.0|1.0     |
+------------------+------------------+------------------+---+--------+



# 1️⃣ Quick debug (optional, to see the issue)

In [11]:
from pyspark.sql import functions as F

print("Train label distribution including nulls:")
(
    train_df.groupBy("Churn_Label")
            .count()
            .orderBy("Churn_Label")
            .show()
)

null_labels = train_df.filter(F.col("Churn_Label").isNull()).count()
print("Rows with NULL Churn_Label in train_df:", null_labels)


Train label distribution including nulls:
+-----------+------+
|Churn_Label| count|
+-----------+------+
|       NULL| 76429|
|          0|159495|
|          1| 68818|
+-----------+------+

Rows with NULL Churn_Label in train_df: 76429


# 💾 Cell 8 – Save the best model to golddata

Here we’ll pick the best model by AUC ROC (you can adjust if you prefer F1).

In [19]:
# Cell 8 — Pick best model and save it to golddata/models/policy_churn

# Choose best by AUC ROC
best_name, best_metrics = max(results, key=lambda t: t[1]["auc_roc"])
print(f"Best model by AUC ROC: {best_name} → {best_metrics}")

if best_name == "LogisticRegression":
    best_model = lr_model
elif best_name == "RandomForest":
    best_model = rf_model
else:
    best_model = gbt_model

MODEL_BASE_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"models/policy_churn"
)

MODEL_PATH = f"{MODEL_BASE_PATH}/{best_name.lower()}_v1"

print("Saving best model to:", MODEL_PATH)

(best_model.write()
           .overwrite()
           .save(MODEL_PATH))

print("✅ Model saved.")


Best model by AUC ROC: RandomForest → {'auc_roc': 1.0, 'auc_pr': 1.0, 'f1': 1.0, 'accuracy': 1.0}
Saving best model to: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/policy_churn/randomforest_v1


✅ Model saved.


# 🔍 Cell 9 – Score a few policies as a “serving” simulation

In [22]:
# Cell 9 — Load best model and score a few policies (serving simulation)

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

# 1) Load the saved best model (path from Cell 8)
loaded_model = PipelineModel.load(MODEL_PATH)

# 2) Take a small random sample to mimic online scoring
sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

# 3) Apply same preprocessing + model
scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

print("Scored schema:")
scored_raw.printSchema()

# 4) Convert probability vector -> array and pick P(churn=1)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .select(
            "Policy_ID",
            "Customer_ID",
            "Churn_Label",
            F.col("prob_arr")[1].alias("churn_prob"),   # probability of class 1
            "prediction",
            "Product_Line",
            "Channel",
            "Annual_Premium_GBP",
            "Tenure_Band",
            "Discount_Band"
        )
)

scored.show(truncate=False)


Scored schema:
root
 |-- Policy_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Line: string (nullable = false)
 |-- Channel: string (nullable = false)
 |-- Sum_Insured_GBP: double (nullable = false)
 |-- Annual_Premium_GBP: double (nullable = false)
 |-- Policy_Start_Date: date (nullable = true)
 |-- Policy_End_Date: date (nullable = true)
 |-- Policy_Duration_Days: double (nullable = false)
 |-- Renewal_Offered_Flag: integer (nullable = true)
 |-- Renewal_Accepted_Flag: integer (nullable = true)
 |-- Renewal_Conversion: integer (nullable = true)
 |-- Tenure_Band: string (nullable = false)
 |-- Premium_Band: string (nullable = false)
 |-- Discount_Band: string (nullable = false)
 |-- Renewal_Outcome: string (nullable = false)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_discount_valid: integer (nullable = true)
 |-- dq_renewal_valid: integer (nullable = true)
 |-- Is_Renewal_Offered: integer (nullable = true)
 |-- Churn_Label: double (nul

# 📈 Cell 10 – (Optional) Write evaluation metrics to a gold metrics table

If you want the same metrics logging behaviour you used elsewhere:

In [23]:
# Cell 10 — OPTIONAL: log metrics into a Delta metrics table in gold

from datetime import datetime

METRICS_PATH = (
    f"abfss://{CONTAINER_GOLD}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"_metrics_ml_policy_churn"
)

now_ts = datetime.utcnow()

metrics_log_rows = [
    ("policy_churn", best_name, k, float(v), now_ts)
    for k, v in best_metrics.items()
]

metrics_log_df = spark.createDataFrame(
    metrics_log_rows,
    "model_family STRING, model_name STRING, metric STRING, value DOUBLE, ts TIMESTAMP"
)

(metrics_log_df
    .write
    .format("delta")
    .mode("append")
    .save(METRICS_PATH))

print("✅ Logged metrics to:", METRICS_PATH)

spark.read.format("delta").load(METRICS_PATH).orderBy(F.col("ts").desc()).show()


✅ Logged metrics to: abfss://golddata@clientdatastorage.dfs.core.windows.net/_metrics_ml_policy_churn
+------------+------------+--------+-----+--------------------+
|model_family|  model_name|  metric|value|                  ts|
+------------+------------+--------+-----+--------------------+
|policy_churn|RandomForest|      f1|  1.0|2025-12-07 19:16:...|
|policy_churn|RandomForest| auc_roc|  1.0|2025-12-07 19:16:...|
|policy_churn|RandomForest|accuracy|  1.0|2025-12-07 19:16:...|
|policy_churn|RandomForest|  auc_pr|  1.0|2025-12-07 19:16:...|
+------------+------------+--------+-----+--------------------+

